# Lifecycle + price-availability features experiment

Diagnostic step 1 found that price features are highly important but ~19% of price rows are missing. This experiment adds explicit lifecycle / price-availability features so the model can distinguish *true* zero demand from *pre-launch* / *inactive* / *missing-price* state.

## Features added (Phase 4 schema bump 54 -> 64)

Origin-side (9 columns):
* `has_price` -- today's price is known at origin
* `days_since_first_sale` / `days_since_last_sale` -- shift(1)-safe sales-based features
* `days_since_first_price` / `days_since_last_price` -- inclusive of today (price is known)
* `is_active_after_first_sale` / `is_active_after_first_price`
* `pre_first_sale_flag` / `pre_first_price_flag`

Target-side (1 column):
* `target_has_price`

## Leakage rules
* Sales-based features use `groupby('id').shift(1)` -- row at date `t` never sees `sales[t]`.
* Price-based features use today's price (consistent with how `sell_price` and `price_rolling_mean_28` are handled in Phase 4).

## Outputs

Trained artifacts go to:
```
outputs/models/experiments/lifecycle_features/
outputs/reports/experiments/lifecycle_features/
outputs/figures/experiments/lifecycle_features/
```

The pre-experiment baseline artifacts in `outputs/reports/lightgbm_*` and `outputs/reports/quantile_*` are preserved as the comparison snapshot.

## Order of operations

1. Rebuild features (the supervised parquets now include lifecycle columns).
2. Run the experiment (trains LGBM point + quantile, runs diagnostics, writes before-vs-after CSV).

```python
# !python -m seercast.training.build_features          # one-time, after the schema bump
# !python -m seercast.training.run_lifecycle_experiment
```

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / 'src'))

import pandas as pd
from seercast.training.run_lifecycle_experiment import run as run_experiment
from seercast.config import ARTIFACTS, REPORTS_DIR

pd.options.display.float_format = '{:.4f}'.format
REPO_ROOT

## 1. Run the experiment

This trains both LGBM models on the lifecycle-augmented feature set, runs the diagnostics layer, and writes the before-vs-after CSV.

In [ ]:
result = run_experiment()
paths = result['paths']
before_vs_after = result['before_vs_after']
before_vs_after

## 2. Did WAPE improve?

In [ ]:
pivot_wape = before_vs_after.pivot(index='model', columns='version', values='WAPE')
pivot_wape['delta_wape'] = pivot_wape['lifecycle_features'] - pivot_wape['baseline_features']
pivot_wape

## 3. Did the quantile p50 negative bias shrink?

In [ ]:
pivot_bias = before_vs_after.pivot(index='model', columns='version', values='Bias')
pivot_bias['delta_bias'] = pivot_bias['lifecycle_features'] - pivot_bias['baseline_features']
pivot_bias

## 4. RMSE trade-off?

In [ ]:
before_vs_after.pivot(index='model', columns='version', values='RMSE')

## 5. Where did improvements concentrate? (segment + horizon)

The experiment's diagnostics directory has per-segment, per-horizon, per-is_active breakdowns recomputed on the new features.

In [ ]:
diag_dir = paths['diag_reports_dir']
by_seg_new = pd.read_csv(diag_dir / 'error_breakdown_by_segment_at_origin.csv')
by_seg_old = pd.read_csv(REPORTS_DIR / 'diagnostics' / 'error_breakdown_by_segment_at_origin.csv')
by_seg_new.set_index(['segment_at_origin', 'model'])['WAPE'].sub(
    by_seg_old.set_index(['segment_at_origin', 'model'])['WAPE']
).rename('delta_WAPE').to_frame().sort_values('delta_WAPE')

## 6. Did lifecycle features appear in top importance?

In [ ]:
fi = pd.read_csv(diag_dir / 'feature_importance_combined.csv')
lifecycle_cols = ['has_price', 'days_since_first_sale', 'days_since_last_sale',
                  'days_since_first_price', 'days_since_last_price',
                  'is_active_after_first_sale', 'is_active_after_first_price',
                  'pre_first_sale_flag', 'pre_first_price_flag']
fi['is_lifecycle'] = fi['feature'].isin(lifecycle_cols)
print('top 25 features by point_gain:')
fi.head(25)

## 7. Interpretation

Whatever the result shows, we report it. The questions:

1. Did WAPE improve beyond the previous best (`lightgbm_quantile_p50 = 0.697314`)?
2. Did the p50 negative bias shrink (closer to 0 than -0.17)?
3. Did RMSE move (better/worse than before)?
4. Did raw `price_change_*` features drop in importance, replaced by lifecycle features?
5. Are improvements concentrated in `zero_heavy` / `intermittent` / `pre-launch` segments?

If the answer to (1)+(2) is yes, the experiment is a win. If only some are yes, it's a partial win. If neither moves, lifecycle features alone weren't the bottleneck and we look elsewhere next (objective sweep, per-horizon models, etc.).